# Week 2 – Data Preprocessing in Python
**Capstone: Smart Home Energy Usage Tracker**

## Setup

In [1]:
import pandas as pd
import numpy as np

## Load Dataset

In [2]:
from google.colab import files
uploaded = files.upload()

Saving energy_usage.csv to energy_usage.csv


In [3]:
df = pd.read_csv("energy_usage.csv")
print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print("\nRaw Data Preview:")
print(df.head(10).to_string(index=False))

Loaded 77 rows, 8 columns

Raw Data Preview:
 log_id device_id     device_name room_id    room_name  energy_kwh           timestamp status
      1      D001 Air Conditioner    R001  Living Room        1.82 2025-04-01 00:00:00 active
      2      D002    Refrigerator    R002      Kitchen        0.45 2025-04-01 01:00:00 active
      3      D003 Washing Machine    R003 Utility Room        0.91 2025-04-01 02:00:00   idle
      4      D004      Television    R001  Living Room        0.12 2025-04-01 03:00:00 active
      5      D005       Microwave    R002      Kitchen        0.08 2025-04-01 04:00:00   idle
      6      D006    Water Heater    R004     Bathroom        1.20 2025-04-01 05:00:00 active
      7      D007      Dishwasher    R002      Kitchen        0.65 2025-04-01 06:00:00   idle
      8      D008  Laptop Charger    R005      Bedroom        0.05 2025-04-01 07:00:00 active
      9      D009     Ceiling Fan    R001  Living Room        0.07 2025-04-01 08:00:00 active
     10      D0

## Clean Data

In [4]:
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["energy_kwh"] = pd.to_numeric(df["energy_kwh"], errors="coerce")
df["device_name"] = df["device_name"].str.strip().str.title()
df["room_name"] = df["room_name"].str.strip().str.title()
df["status"] = df["status"].str.strip().str.lower()

bad_kwh = df[df["energy_kwh"].isna()]
if not bad_kwh.empty:
    print(f"Found {len(bad_kwh)} rows with invalid energy values — dropping.")
df.dropna(subset=["energy_kwh"], inplace=True)

df["date"] = df["timestamp"].dt.date
df["hour"] = df["timestamp"].dt.hour
df["month"] = df["timestamp"].dt.to_period("M")
df.reset_index(drop=True, inplace=True)

print(f"Rows after cleaning: {len(df)}")
print("\nCleaned Data Sample:")
print(df[["device_name", "room_name", "energy_kwh", "timestamp", "status"]].head(10).to_string(index=False))

Rows after cleaning: 77

Cleaned Data Sample:
    device_name    room_name  energy_kwh           timestamp status
Air Conditioner  Living Room        1.82 2025-04-01 00:00:00 active
   Refrigerator      Kitchen        0.45 2025-04-01 01:00:00 active
Washing Machine Utility Room        0.91 2025-04-01 02:00:00   idle
     Television  Living Room        0.12 2025-04-01 03:00:00 active
      Microwave      Kitchen        0.08 2025-04-01 04:00:00   idle
   Water Heater     Bathroom        1.20 2025-04-01 05:00:00 active
     Dishwasher      Kitchen        0.65 2025-04-01 06:00:00   idle
 Laptop Charger      Bedroom        0.05 2025-04-01 07:00:00 active
    Ceiling Fan  Living Room        0.07 2025-04-01 08:00:00 active
    Room Heater      Bedroom        1.50 2025-04-01 09:00:00 active


## Overall Statistics (NumPy)

In [5]:
kwh = df["energy_kwh"].to_numpy()

print("Overall Statistics (NumPy):")
print(f"  Total Energy    : {np.sum(kwh):.3f} kWh")
print(f"  Mean per Log    : {np.mean(kwh):.3f} kWh")
print(f"  Median          : {np.median(kwh):.3f} kWh")
print(f"  Std Deviation   : {np.std(kwh):.3f} kWh")
print(f"  Min             : {np.min(kwh):.3f} kWh")
print(f"  Max             : {np.max(kwh):.3f} kWh")

Overall Statistics (NumPy):
  Total Energy    : 56.050 kWh
  Mean per Log    : 0.728 kWh
  Median          : 0.460 kWh
  Std Deviation   : 0.678 kWh
  Min             : 0.000 kWh
  Max             : 2.200 kWh


## Energy per Device (NumPy)

In [6]:
print("Energy per Device:")
for device in df["device_name"].unique():
    vals = df[df["device_name"] == device]["energy_kwh"].to_numpy()
    print(f"  {device:<20} Total: {np.sum(vals):.3f} kWh  |  Avg: {np.mean(vals):.3f} kWh")

Energy per Device:
  Air Conditioner      Total: 19.500 kWh  |  Avg: 1.950 kWh
  Refrigerator         Total: 4.030 kWh  |  Avg: 0.448 kWh
  Washing Machine      Total: 5.430 kWh  |  Avg: 0.776 kWh
  Television           Total: 1.080 kWh  |  Avg: 0.135 kWh
  Microwave            Total: 0.690 kWh  |  Avg: 0.099 kWh
  Water Heater         Total: 9.680 kWh  |  Avg: 1.210 kWh
  Dishwasher           Total: 4.110 kWh  |  Avg: 0.587 kWh
  Laptop Charger       Total: 0.350 kWh  |  Avg: 0.050 kWh
  Ceiling Fan          Total: 0.500 kWh  |  Avg: 0.071 kWh
  Room Heater          Total: 10.680 kWh  |  Avg: 1.526 kWh


## Device-Level Summary (Pandas)

In [7]:
device_summary = (
    df.groupby(["device_id", "device_name"])["energy_kwh"]
    .agg(total_kwh="sum", avg_kwh="mean", num_logs="count")
    .round(3)
    .reset_index()
    .sort_values("total_kwh", ascending=False)
)
print("Device-Level Summary:")
print(device_summary.to_string(index=False))

Device-Level Summary:
device_id     device_name  total_kwh  avg_kwh  num_logs
     D001 Air Conditioner      19.50    1.950        10
     D010     Room Heater      10.68    1.526         7
     D006    Water Heater       9.68    1.210         8
     D003 Washing Machine       5.43    0.776         7
     D007      Dishwasher       4.11    0.587         7
     D002    Refrigerator       4.03    0.448         9
     D004      Television       1.08    0.135         8
     D005       Microwave       0.69    0.099         7
     D009     Ceiling Fan       0.50    0.071         7
     D008  Laptop Charger       0.35    0.050         7


## Room-Level Summary (Pandas)

In [8]:
room_summary = (
    df.groupby("room_name")["energy_kwh"]
    .agg(total_kwh="sum", avg_kwh="mean", num_logs="count")
    .round(3)
    .reset_index()
    .sort_values("total_kwh", ascending=False)
)
print("Room-Level Summary:")
print(room_summary.to_string(index=False))

Room-Level Summary:
   room_name  total_kwh  avg_kwh  num_logs
 Living Room      21.08    0.843        25
     Bedroom      11.03    0.788        14
    Bathroom       9.68    1.210         8
     Kitchen       8.83    0.384        23
Utility Room       5.43    0.776         7


## High Usage Alert

In [9]:
daily_summary = (
    df.groupby(["device_name", "date"])["energy_kwh"]
    .sum()
    .reset_index()
    .rename(columns={"energy_kwh": "daily_kwh"})
)

threshold = 10.0
high_usage = daily_summary[daily_summary["daily_kwh"] > threshold]

print(f"Devices Exceeding {threshold} kWh in a Day:")
if high_usage.empty:
    print("  None found.")
else:
    print(high_usage.to_string(index=False))

Devices Exceeding 10.0 kWh in a Day:
  None found.


## Save Reports

In [10]:
device_summary.to_csv("device_summary.csv", index=False)
room_summary.to_csv("room_summary.csv", index=False)
print("Reports Saved: device_summary.csv, room_summary.csv")
print("Pipeline Completed")

Reports Saved: device_summary.csv, room_summary.csv
Pipeline Completed
